# Car Price Prediction and Recommendation System
This notebook demonstrates data cleaning, modeling, and deployment for car price prediction and recommendation using the provided `output.csv` dataset.

In [ ]:
# 1. Import Libraries and Set Configuration
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os
import datetime

# File paths
DATA_PATH = 'output.csv'  # Use local file
CLEANED_DATA_SAVE_PATH = 'cleaned_data_for_deployment_updated.csv'
BEST_XGB_MODEL_PATH = 'best_xgb_model_updated.joblib'
LINEAR_REG_MODEL_PATH = 'linear_reg_model_updated.joblib'
RF_MODEL_PATH = 'rf_model_updated.joblib'
SCALER_REC_PATH = 'scaler_rec_updated.joblib'

In [ ]:
# Load Data from CSV
import pandas as pd

df = pd.read_csv('D:\data_scrapping_pakwheels\data\processed\enriched_latest.csv')
df.head()

In [ ]:
# Display DataFrame columns, first few rows, and missing value counts for debugging
print("\n--- DataFrame Columns and Sample Rows ---")
print(df_cleaned.columns.tolist())
display(df_cleaned.head())

print("\n--- Missing Value Counts for Key Columns ---")
required_cols = ['modelDate', 'mileageFromOdometer', 'engineDisplacement', 'fuelType', 'seller_location', 'price']
print(df_cleaned[required_cols].isnull().sum())
print("\nTotal rows in cleaned data:", len(df_cleaned))

In [ ]:
# Data Cleaning and Preprocessing
print("\n--- Data Cleaning and Preprocessing ---")
df_cleaned = None
try:
    # Handle duplicates
    initial_rows = df.shape[0]
    df_cleaned = df.drop_duplicates().copy()
    rows_after_dropping_duplicates = df_cleaned.shape[0]
    print(f"Rows before dropping duplicates: {initial_rows}")
    print(f"Rows after dropping duplicates: {rows_after_dropping_duplicates}")
    print(f"Duplicate rows removed: {initial_rows - rows_after_dropping_duplicates}")
    # Convert price to numeric, coerce errors to NaN, then drop missing
    df_cleaned['price'] = pd.to_numeric(df_cleaned['price'], errors='coerce')
    df_cleaned = df_cleaned[df_cleaned['price'].notna()]
    print(f"Rows after dropping missing/non-numeric price: {df_cleaned.shape[0]}")
    # Robustly handle numeric columns
    for col in ['modelDate', 'mileageFromOdometer', 'engineDisplacement']:
        if col in df_cleaned.columns:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
            if df_cleaned[col].notna().sum() == 0:
                print(f"Column '{col}' has all NaN values. Dropping column.")
                df_cleaned = df_cleaned.drop(columns=[col])
            elif df_cleaned[col].isnull().any():
                median_val = df_cleaned[col].median()
                print(f"Filling missing values in '{col}' with median: {median_val}")
                df_cleaned[col] = df_cleaned[col].fillna(median_val)
        else:
            print(f"Column '{col}' not found in data. Skipping.")
    # Fill missing values for categorical features
    for col in ['fuelType', 'seller_location']:
        if col in df_cleaned.columns:
            if df_cleaned[col].isnull().any():
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    fill_val = mode_val[0]
                    print(f"Filling missing values in '{col}' with mode: {fill_val}")
                    df_cleaned[col] = df_cleaned[col].fillna(fill_val)
                else:
                    print(f"Column '{col}' has all NaN values. Filling with 'Unknown'.")
                    df_cleaned[col] = df_cleaned[col].fillna('Unknown')
        else:
            print(f"Categorical column '{col}' not found in data. Skipping.")
except Exception as e:
    import traceback
    print(f"An error occurred during data cleaning and preprocessing: {repr(e)}")
    traceback.print_exc()
    df_cleaned = None

if df_cleaned is None or df_cleaned.empty:
    print("Exiting due to data cleaning/preprocessing error or empty cleaned data.")
    # Optionally, raise an error or stop execution in a script

In [ ]:
# Feature Engineering and Data Preparation
print("\n--- Preparing Data for Modeling ---")
from sklearn.preprocessing import OneHotEncoder

# Define features based on your columns (modelDate removed)
numerical_features = ['mileageFromOdometer', 'engineDisplacement']
categorical_features = ['fuelType', 'seller_location']
all_features = numerical_features + categorical_features

target_column = 'price'

# Drop rows with missing values in required columns
df_model = df_cleaned.dropna(subset=all_features + [target_column]).copy()

# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(df_model[categorical_features])
encoded_cat_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(categorical_features), index=df_model.index)

# Combine numerical and encoded categorical features
X = pd.concat([df_model[numerical_features], encoded_cat_df], axis=1)
y = df_model[target_column]

print(f"Prepared feature matrix X shape: {X.shape}")
print(f"Prepared target vector y shape: {y.shape}")

# Print all feature names used in the model input
print("Model input features:")
print(list(X.columns))

In [ ]:
# Train-Test Split and Feature Scaling
print("\n--- Train-Test Split and Feature Scaling ---")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Feature scaling (important for some models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

In [ ]:
# Build and Train Price Prediction Models
print("\n--- Building and Training Price Prediction Models ---")
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Linear Regression
print("Training Linear Regression model...")
linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train_scaled, y_train)
print("Linear Regression model trained.")

# Random Forest
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train_scaled, y_train)
print("Random Forest Regressor trained.")

# XGBoost
print("Training XGBoost Regressor...")
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train_scaled, y_train)
print("XGBoost Regressor trained.")

In [ ]:
# Evaluate Price Prediction Models
print("\n--- Evaluating Price Prediction Models ---")
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions
y_pred_linear = linear_reg_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_xgb = xgb_model.predict(X_test_scaled)

# Linear Regression metrics
mae_linear = mean_absolute_error(y_test, y_pred_linear)
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)
print("--- Linear Regression Model Performance ---")
print(f"MAE: {mae_linear:.4f}")
print(f"MSE: {mse_linear:.4f}")
print(f"R2: {r2_linear:.4f}")
print("-" * 40)

# Random Forest metrics
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print("--- Random Forest Regressor Model Performance ---")
print(f"MAE: {mae_rf:.4f}")
print(f"MSE: {mse_rf:.4f}")
print(f"R2: {r2_rf:.4f}")
print("-" * 40)

# XGBoost metrics
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
print("--- XGBoost Regressor Model Performance ---")
print(f"MAE: {mae_xgb:.4f}")
print(f"MSE: {mse_xgb:.4f}")
print(f"R2: {r2_xgb:.4f}")
print("-" * 40)

In [ ]:
# Hyperparameter Tuning for XGBoost
print("\n--- Hyperparameter Tuning for XGBoost ---")
from sklearn.model_selection import GridSearchCV

xgb_tune = XGBRegressor(random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.1],
    'max_depth': [3],
    'subsample': [1.0],
    'colsample_bytree': [1.0]
}
grid_search = GridSearchCV(
    estimator=xgb_tune,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    verbose=2,
    n_jobs=-1
)
print("Starting GridSearchCV to find best hyperparameters...")
grid_search.fit(X_train_scaled, y_train)
print("GridSearchCV finished.")

print("\nBest hyperparameters found:")
print(grid_search.best_params_)

In [ ]:
# Final XGBoost Model Training and Evaluation
print("\n--- Final XGBoost Model Training and Evaluation ---")
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train final model with best params
y_best_params = grid_search.best_params_
best_xgb_model = XGBRegressor(**y_best_params, random_state=42)
best_xgb_model.fit(X_train_scaled, y_train)

y_pred_best_xgb = best_xgb_model.predict(X_test_scaled)

# Evaluate the tuned model
mae_best_xgb = mean_absolute_error(y_test, y_pred_best_xgb)
mse_best_xgb = mean_squared_error(y_test, y_pred_best_xgb)
r2_best_xgb = r2_score(y_test, y_pred_best_xgb)

print("\n--- Tuned XGBoost Regressor Model Performance ---")
print(f"MAE: {mae_best_xgb:.4f}")
print(f"MSE: {mse_best_xgb:.4f}")
print(f"R2: {r2_best_xgb:.4f}")
print("-" * 40)

In [25]:
# Build Car Recommendation System
print("\n--- Building Car Recommendation System ---")
from sklearn.metrics.pairwise import cosine_similarity

# Use only available features
recommendation_features = ['mileageFromOdometer', 'engineDisplacement', 'fuelType', 'seller_location']

# Ensure all features exist in the cleaned data
missing_cols = [col for col in recommendation_features if col not in df_cleaned.columns]
if missing_cols:
    print(f"Warning: The following columns are missing and will be skipped: {missing_cols}")
    recommendation_features = [col for col in recommendation_features if col in df_cleaned.columns]

if len(recommendation_features) < 2:
    print("Not enough features available for recommendation system.")
else:
    df_rec = df_cleaned.dropna(subset=recommendation_features).copy()
    # One-hot encode categorical features for recommendation
    rec_cat_features = [col for col in ['fuelType', 'seller_location'] if col in df_rec.columns]
    rec_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    rec_cat = rec_encoder.fit_transform(df_rec[rec_cat_features])
    rec_cat_df = pd.DataFrame(rec_cat, columns=rec_encoder.get_feature_names_out(rec_cat_features), index=df_rec.index)
    # Combine numerical and encoded categorical features
    rec_num_features = [col for col in ['mileageFromOdometer', 'engineDisplacement'] if col in df_rec.columns]
    rec_X = pd.concat([df_rec[rec_num_features], rec_cat_df], axis=1)
    # Scale features
    rec_scaler = StandardScaler()
    rec_X_scaled = rec_scaler.fit_transform(rec_X)
    # Compute similarity matrix
    similarity_matrix = cosine_similarity(rec_X_scaled)
    print(f"Similarity matrix shape: {similarity_matrix.shape}")
    # Example: Recommend top 5 similar cars for the first car
    car_idx = 0
    similar_indices = similarity_matrix[car_idx].argsort()[::-1][1:6]
    print(f"Top 5 similar cars to car at index {car_idx}: {similar_indices}")

Similarity matrix shape: (66243, 66243)
Top 5 similar cars to car at index 0: [47924 47445 30301 47286 21863]


In [ ]:
# Save Models, Scaler, and Cleaned Data for Deployment
print("\n--- Saving Models, Scaler, and Cleaned Data for Deployment ---")
import joblib

try:
    # Save models
    joblib.dump(best_xgb_model, BEST_XGB_MODEL_PATH)
    print(f"Tuned XGBoost Model saved to: {BEST_XGB_MODEL_PATH}")
    joblib.dump(linear_reg_model, LINEAR_REG_MODEL_PATH)
    print(f"Linear Regression Model saved to: {LINEAR_REG_MODEL_PATH}")
    joblib.dump(rf_model, RF_MODEL_PATH)
    print(f"Random Forest Model saved to: {RF_MODEL_PATH}")
    # Save scaler for recommendation system
    joblib.dump(rec_scaler, SCALER_REC_PATH)
    print(f"Scaler for recommendation system saved to: {SCALER_REC_PATH}")
except Exception as e:
    print(f"Error saving models or scaler: {e}")

# Save cleaned data
df_cleaned.to_csv(CLEANED_DATA_SAVE_PATH, index=False)
print(f"Cleaned data saved to: {CLEANED_DATA_SAVE_PATH}")

In [ ]:
# Download Saved Files (Local Save Only)
import os

files_to_save = [
    BEST_XGB_MODEL_PATH,
    LINEAR_REG_MODEL_PATH,
    RF_MODEL_PATH,
    SCALER_REC_PATH,
    CLEANED_DATA_SAVE_PATH
]

print("The following files have been saved in your working directory:")
for file_name in files_to_save:
    if os.path.exists(file_name):
        print(f"- {file_name}")
    else:
        print(f"- {file_name} (not found, may not have been saved successfully)")
print("\nYou can manually access these files in your local directory.")

In [ ]:
# 2. Clean and Preprocess New Data, Store with Timestamp
import pandas as pd
import datetime
import os

def clean_and_store_latest_data(raw_data_path):
    df = pd.read_csv(raw_data_path)
    # --- Use robust cleaning logic from earlier cells ---
    df_cleaned = df.drop_duplicates().copy()
    df_cleaned['price'] = pd.to_numeric(df_cleaned['price'], errors='coerce')
    df_cleaned = df_cleaned[df_cleaned['price'].notna()]
    for col in ['modelDate', 'mileageFromOdometer', 'engineDisplacement']:
        if col in df_cleaned.columns:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
            if df_cleaned[col].notna().sum() == 0:
                df_cleaned = df_cleaned.drop(columns=[col])
            elif df_cleaned[col].isnull().any():
                df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
    for col in ['fuelType', 'seller_location']:
        if col in df_cleaned.columns:
            if df_cleaned[col].isnull().any():
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])
                else:
                    df_cleaned[col] = df_cleaned[col].fillna('Unknown')
    # Save with timestamp
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    cleaned_path = f"data/cleaned_{ts}.csv"
    df_cleaned.to_csv(cleaned_path, index=False)
    print(f"Cleaned data saved to: {cleaned_path}")
    return cleaned_path

# Example usage:
# latest_cleaned = clean_and_store_latest_data('data/raw_YYYYMMDD.csv')

In [ ]:
# 3. Hyperparameter Tuning for All Models on Updated Data
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def tune_models(X_train, y_train):
    # Random Forest
    rf_params = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
    rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=3, scoring='r2', n_jobs=-1)
    rf_grid.fit(X_train, y_train)
    print("Best RF params:", rf_grid.best_params_)

    # XGBoost
    xgb_params = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
    xgb_grid = GridSearchCV(XGBRegressor(random_state=42), xgb_params, cv=3, scoring='r2', n_jobs=-1)
    xgb_grid.fit(X_train, y_train)
    print("Best XGB params:", xgb_grid.best_params_)

    # Linear Regression (no major hyperparams)
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    return rf_grid.best_estimator_, xgb_grid.best_estimator_, lr

# Example usage:
# rf_best, xgb_best, lr_best = tune_models(X_train_scaled, y_train)

In [ ]:
# 4. Evaluate Ensemble Methods (Stacking)
from sklearn.ensemble import StackingRegressor

def build_stacking_ensemble(X_train, y_train):
    estimators = [
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('xgb', XGBRegressor(n_estimators=100, random_state=42))
    ]
    stack = StackingRegressor(
        estimators=estimators,
        final_estimator=LinearRegression(),
        n_jobs=-1
    )
    stack.fit(X_train, y_train)
    print("Stacking ensemble trained.")
    return stack

# Example usage:
# stacking_model = build_stacking_ensemble(X_train_scaled, y_train)

In [ ]:
# 5. Optimize Recommendation Engine: Hybrid Filtering
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def hybrid_recommendation(df, user_history=None, top_n=5):
    # Content-based: cosine similarity as before
    features = ['mileageFromOdometer', 'engineDisplacement', 'fuelType', 'seller_location']
    available = [f for f in features if f in df.columns]
    df_rec = df.dropna(subset=available).copy()
    # One-hot encode categoricals
    cat = [f for f in ['fuelType', 'seller_location'] if f in df_rec.columns]
    enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    cat_df = pd.DataFrame(enc.fit_transform(df_rec[cat]), columns=enc.get_feature_names_out(cat), index=df_rec.index)
    num = [f for f in ['mileageFromOdometer', 'engineDisplacement'] if f in df_rec.columns]
    X = pd.concat([df_rec[num], cat_df], axis=1)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    sim = cosine_similarity(X_scaled)
    # Popularity-based: recommend most frequent models
    pop = df['model'].value_counts().head(top_n).index.tolist() if 'model' in df.columns else []
    # Hybrid: combine top content-based and top popular
    hybrid = []
    if user_history is not None and len(df_rec) > 0:
        idx = 0  # Example: first car in user history
        similar_idx = sim[idx].argsort()[::-1][1:top_n+1]
        hybrid = df_rec.iloc[similar_idx]['model'].tolist() if 'model' in df_rec.columns else []
    hybrid += [m for m in pop if m not in hybrid]
    print(f"Hybrid recommendations: {hybrid[:top_n]}")
    return hybrid[:top_n]

# Example usage:
# hybrid_recommendation(df_cleaned, user_history=[...])

In [ ]:
# 5.1. Enhanced Hybrid Recommendation Engine: User Profile & Diversity
import random

def enhanced_hybrid_recommendation(df, user_history=None, top_n=5, diversity=True):
    features = ['mileageFromOdometer', 'engineDisplacement', 'fuelType', 'seller_location']
    available = [f for f in features if f in df.columns]
    df_rec = df.dropna(subset=available).copy()
    cat = [f for f in ['fuelType', 'seller_location'] if f in df_rec.columns]
    enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    cat_df = pd.DataFrame(enc.fit_transform(df_rec[cat]), columns=enc.get_feature_names_out(cat), index=df_rec.index)
    num = [f for f in ['mileageFromOdometer', 'engineDisplacement'] if f in df_rec.columns]
    X = pd.concat([df_rec[num], cat_df], axis=1)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    sim = cosine_similarity(X_scaled)
    pop = df['model'].value_counts().head(top_n*2).index.tolist() if 'model' in df.columns else []
    hybrid = []
    # User profile: average features from user_history
    if user_history is not None and len(df_rec) > 0:
        if isinstance(user_history, list) and len(user_history) > 0:
            # Find indices of user_history models in df_rec
            user_indices = df_rec[df_rec['model'].isin(user_history)].index.tolist() if 'model' in df_rec.columns else []
            if user_indices:
                user_vec = X_scaled[user_indices].mean(axis=0, keepdims=True)
                sim_scores = cosine_similarity(user_vec, X_scaled).flatten()
                similar_idx = sim_scores.argsort()[::-1][:top_n*2]
                hybrid = df_rec.iloc[similar_idx]['model'].tolist() if 'model' in df_rec.columns else []
    # Add popularity-based if not enough
    hybrid += [m for m in pop if m not in hybrid]
    # Diversity: shuffle and deduplicate
    if diversity:
        seen = set()
        diverse = []
        for m in hybrid:
            if m not in seen:
                diverse.append(m)
                seen.add(m)
            if len(diverse) >= top_n:
                break
        random.shuffle(diverse)
        hybrid = diverse
    print(f"Enhanced hybrid recommendations: {hybrid[:top_n]}")
    return hybrid[:top_n]

# Example usage:
# enhanced_hybrid_recommendation(df_cleaned, user_history=['Civic', 'Corolla'], top_n=5)

## Step 1: Database Integration (PostgreSQL & MongoDB)
This section demonstrates how to connect to PostgreSQL and MongoDB, and how to save/load predictions and recommendations. 
Install the required libraries if not already present:
- `psycopg2-binary` for PostgreSQL
- `pymongo` for MongoDB

In [ ]:
# Install required packages for database integration
%pip install psycopg2-binary pymongo

In [ ]:
# Install required packages (uncomment if needed)
# !pip install psycopg2-binary pymongo

import pandas as pd

# PostgreSQL Example
import psycopg2
from psycopg2.extras import execute_values

def save_predictions_postgres(df, table_name='car_predictions', db_params=None):
    """
    Save DataFrame predictions to PostgreSQL.
    db_params: dict with keys host, dbname, user, password, port
    """
    if db_params is None:
        print("No DB params provided. Skipping PostgreSQL save.")
        return
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()
    # Create table if not exists
    create_table = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id SERIAL PRIMARY KEY,
        model VARCHAR(100),
        predicted_price FLOAT,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    cur.execute(create_table)
    # Insert data
    records = df[['model', 'predicted_price']].values.tolist()
    insert_query = f"INSERT INTO {table_name} (model, predicted_price) VALUES %s"
    execute_values(cur, insert_query, records)
    conn.commit()
    cur.close()
    conn.close()
    print(f"Saved {len(records)} predictions to PostgreSQL table '{table_name}'.")

# MongoDB Example
from pymongo import MongoClient

def save_recommendations_mongo(recommendations, db_name='car_db', collection_name='recommendations', mongo_uri='mongodb://localhost:27017/'):
    """
    Save recommendations (list of dicts) to MongoDB.
    """
    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]
    if isinstance(recommendations, list):
        collection.insert_many([{'recommendation': r} for r in recommendations])
        print(f"Saved {len(recommendations)} recommendations to MongoDB collection '{collection_name}'.")
    else:
        print("Recommendations should be a list.")
    client.close()

# Example usage:
# save_predictions_postgres(df_predictions, db_params={'host':..., 'dbname':..., 'user':..., 'password':..., 'port':5432})
# save_recommendations_mongo(['Civic', 'Corolla'])